# Experiment-1: **DenseNet-121**





In [ ]:
import os
import cv2
import torch
import wandb
import random
import numpy as np
import torch.nn as nn
from glob import glob
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset, ConcatDataset
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold
from google.colab import userdata
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from datetime import datetime, date
today = date.today()
curr_date = today.strftime("%d-%m-%Y")
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import torch.nn.functional as F
from torchvision import models, transforms

In [ ]:
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API")
# wandb.login()


In [ ]:
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:

# ---------------- CONFIG ---------------- #
CONFIG = {
    "epochs": 50,
    "batch_size": 16,
    "lr": 1e-4,
    "k_folds": 5,
    "patience": 9,
    # "img_size": 224,
    "num_classes": 3,
    "dataset_path": "/content/drive/MyDrive/Dataset/castor_v2_224x224",
    "usewandb":False
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

# ---------------- DATASET ---------------- #
class PestDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ---------------- LOAD FILES ---------------- #
def load_dataset(root):
    class_folders = sorted(os.listdir(root))
    files, labels = [], []

    for label, cls in enumerate(class_folders):
        img_files = glob(os.path.join(root, cls, "*"))
        files.extend(img_files)
        labels.extend([label] * len(img_files))

    return files, labels


# ---------------- MODEL ---------------- #
import torch.nn as nn
from torchvision import models

def get_model(num_classes=3):
    model = models.densenet121(pretrained=True)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model



# ---------------- CLASS WEIGHTS ---------------- #
def get_class_weights(labels):
    class_count = np.bincount(labels)
    weights = 1.0 / class_count
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float).to(device)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=False, zero_division=0)

    return acc, prec, rec, f1, total_loss / len(loader), cm, report, all_labels, all_preds

In [ ]:

def plot_cm(cm, class_names, title):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path


def train_fold(fold, train_idx, val_idx, class_names, group_name):
    if CONFIG.get("usewandb"):
        wandb.init(
            project=f"DenseNet121_KFold_{curr_date}",
            name=f"Fold-{fold+1}",
            group=group_name,
            config=CONFIG,
        )

    # Dataset split
    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    model = get_model(CONFIG["num_classes"]).to(device)

    # Criterion
    class_weights = get_class_weights(np.array(labels)[train_idx])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    best_acc = 0
    best_prec = 0
    best_rec = 0
    best_f1 = 0
    best_loss = float('inf')
    patience_counter = 0
    best_epoch = -1

    # For storing data of the best epoch
    best_val_labels, best_val_preds = None, None
    best_train_labels, best_train_preds = None, None
    best_val_cm, best_val_report = None, None # Initialize to None
    best_train_cm, best_train_report = None, None # Initialize to None

    for epoch in range(CONFIG["epochs"]):

        print(f"\n====== EPOCH {epoch+1}/{CONFIG['epochs']} ======")

        # ---------------- TRAIN ---------------- #
        model.train()
        train_loss_sum = 0
        all_train_preds, all_train_labels = [], [] # Accumulate predictions and labels for training metrics

        for imgs, lbls in tqdm(train_loader, desc="Training", leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            preds = outputs.argmax(1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(lbls.cpu().numpy())

        train_loss = train_loss_sum / len(train_loader)
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        train_prec = precision_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_rec = recall_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_f1 = f1_score(all_train_labels, all_train_preds, average="macro", zero_division=0)

        # ---------------- VALIDATION ---------------- #
        val_acc, val_prec, val_rec, val_f1, val_loss, \
        val_cm, val_report, v_labels, v_preds = evaluate(model, val_loader, criterion)
        if CONFIG.get("usewandb"):

            # WandB logs per epoch
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_accuracy": train_acc,
                "train_precision": train_prec,
                "train_recall": train_rec,
                "train_f1": train_f1,
                "val_accuracy": val_acc,
                "val_precision": val_prec,
                "val_recall": val_rec,
                "val_f1": val_f1,
            })
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
        print(f"Train Accuracy: {train_acc:.4f} | Train Loss: {train_loss:.4f}")
        print(f"Test Accuracy: {val_acc:.4f} | Test Loss: {val_loss:.4f}")
        # ---------------- BEST EPOCH CHECK ---------------- #
        if val_acc > best_acc:
            best_acc = val_acc
            best_prec = val_prec
            best_rec = val_rec
            best_f1 = val_f1
            best_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0

            # Save model
            model_path = f"densenet_best_fold{fold}.pth"
            torch.save(model.state_dict(), model_path)
            if CONFIG.get("usewandb"):
                wandb.save(model_path)

            # Store validation info
            best_val_labels, best_val_preds = v_labels, v_preds
            best_val_cm, best_val_report = val_cm, val_report

            # Store training info for the best epoch
            best_train_labels, best_train_preds = all_train_labels, all_train_preds
            best_train_cm = confusion_matrix(best_train_labels, best_train_preds)
            best_train_report = classification_report(best_train_labels, best_train_preds, output_dict=False, zero_division=0)

        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print("EARLY STOPPING!")
                break

    # ---------------- LOGGING BEST EPOCH DATA ---------------- #

    print(f"\n===== BEST EPOCH = {best_epoch} | Best Val Acc = {best_acc:.4f} =====")

    # Generate heatmaps - Ensure best_val_cm and best_train_cm are not None
    if best_val_cm is not None and best_train_cm is not None:
        val_cm_img = plot_cm(best_val_cm, class_names, f"Fold{fold}_Validation_CM")
        train_cm_img = plot_cm(best_train_cm, class_names, f"Fold{fold}_Training_CM")
        if CONFIG.get("usewandb"):
            # Log confusion matrices as images
            wandb.log({
                "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                "best_training_confusion_matrix": wandb.Image(train_cm_img),
                "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
            })
    else:
        print("Warning: No best model saved or metrics to log for best epoch.")

    if CONFIG.get("usewandb"):
        wandb.finish()

    return best_acc, best_loss, best_prec, best_rec, best_f1


In [ ]:
# ---------------- MAIN ---------------- #

if 'run' not in globals():
    run = 1

if __name__ == "__main__":
    # Preprocessing (ImageNet, no augmentation)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    files, labels = load_dataset(CONFIG["dataset_path"])
    class_names = sorted(os.listdir(CONFIG["dataset_path"]))
    dataset = PestDataset(files, labels, transform)
    # Store best validation metrics of each fold
    all_fold_val_acc = []
    all_fold_val_loss = []
    all_fold_val_prec = []
    all_fold_val_rec = []
    all_fold_val_f1 = []

    kf = KFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(files)):
        print(f"\n\n========== FOLD {fold+1} ==========")
        group_name = "DenseNet121_KFold_" + f"Run-{run}"
        best_acc, best_loss, best_prec, best_rec, best_f1 = train_fold(
        fold, train_idx, val_idx, class_names, group_name)
        all_fold_val_acc.append(best_acc)
        all_fold_val_loss.append(best_loss)
        all_fold_val_prec.append(best_prec)
        all_fold_val_rec.append(best_rec)
        all_fold_val_f1.append(best_f1)
    run+=1
    print("\n====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======")
    print(f"Average Validation Accuracy : {np.mean(all_fold_val_acc):.4f}")
    print(f"Average Validation Loss     : {np.mean(all_fold_val_loss):.4f}")
    print(f"Average Validation Precision: {np.mean(all_fold_val_prec):.4f}")
    print(f"Average Validation Recall   : {np.mean(all_fold_val_rec):.4f}")
    print(f"Average Validation F1 Score : {np.mean(all_fold_val_f1):.4f}")




========== FOLD 1 ==========


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 202MB/s]



====== EPOCH 1/50 ======


...

Epoch 10/50
Train Accuracy: 0.9975 | Train Loss: 0.0128
Test Accuracy: 0.9697 | Test Loss: 0.0674

====== EPOCH 11/50 ======


Epoch 11/50
Train Accuracy: 0.9962 | Train Loss: 0.0109
Test Accuracy: 0.9697 | Test Loss: 0.0679

====== EPOCH 12/50 ======


Epoch 12/50
Train Accuracy: 0.9975 | Train Loss: 0.0082
Test Accuracy: 0.9697 | Test Loss: 0.0639

====== EPOCH 13/50 ======


Epoch 13/50
Train Accuracy: 0.9962 | Train Loss: 0.0101
Test Accuracy: 0.9697 | Test Loss: 0.0730
EARLY STOPPING!

===== BEST EPOCH = 4 | Best Val Acc = 0.9798 =====


epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
train_accuracy,▁▇████▇██████
train_f1,▁▇████▇██████
train_loss,█▂▂▁▂▁▁▁▁▁▁▁▁
train_precision,▁▇████▇██████
train_recall,▁▇▇███▇██████
val_accuracy,▁▇▇█▆▆▃▃▅▆▆▆▆
val_f1,▁▇▇█▅▅▃▃▄▅▅▅▅
val_loss,█▃▁▁▅▅▄▇▃▂▂▂▃
val_precision,▁▇▇█▆▇▃▅▅▆▆▆▆
+1,...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
